# **Manufacturing Downtime Dataset Cleaning**

Import Required Libraries

In [11]:
import pandas as pd

Load CSV Files

In [12]:
dim_failures = pd.read_csv('Dim_Failure_Types.csv')
dim_machines = pd.read_csv('Dim_Machines.csv')
fact_maintenance = pd.read_csv('Fact_Maintenance_Events.csv')
fact_production = pd.read_csv('Fact_Production_Log.csv')

Create General Cleaning Function

In [13]:
def clean_dataframe(df):
    # Remove leading and trailing whitespace from string columns
    df = df.map(
        lambda x: x.strip() if isinstance(x, str) else x
    )
    # Drop completely identical duplicate rows
    df = df.drop_duplicates()
    return df

Apply General Cleaning to All Datasets

In [14]:
dim_failures = clean_dataframe(dim_failures)
dim_machines = clean_dataframe(dim_machines)
fact_maintenance = clean_dataframe(fact_maintenance)
fact_production = clean_dataframe(fact_production)

Clean Failure Types Dimension

In [15]:
# Fill missing severity and root cause for 'No Failure' cases (F000)
dim_failures["Severity_Level"] = dim_failures["Severity_Level"].fillna("None")
dim_failures["Root_Cause_Category"] = dim_failures["Root_Cause_Category"].fillna("None")

Clean Machines Dimension

In [16]:
# Convert installation date to datetime objects
dim_machines["Installation_Date"] = pd.to_datetime(
    dim_machines["Installation_Date"],
    errors="coerce"
)

Clean Maintenance Events Fact Table

In [17]:
# Convert maintenance date to datetime objects
fact_maintenance["Maintenance_Date"] = pd.to_datetime(
    fact_maintenance["Maintenance_Date"],
    errors="coerce"
)

# Replace missing values in parts replaced with a descriptive string
fact_maintenance["Parts_Replaced"] = fact_maintenance["Parts_Replaced"].fillna("Not Specified")

# Ensure cost is numeric and replace missing values with 0
fact_maintenance["Maintenance_Cost"] = (
    pd.to_numeric(fact_maintenance["Maintenance_Cost"], errors="coerce")
    .fillna(0)
)

Clean Production Log Fact Table

In [18]:
# A. Handle Dates: Convert and drop rows with missing dates (6 rows identified)
fact_production["Date"] = pd.to_datetime(fact_production["Date"], errors="coerce")
fact_production = fact_production.dropna(subset=["Date"])

# B. Logical Integrity: Filter out rows where Downtime > Planned Time (4 rows identified)
fact_production = fact_production[fact_production["Downtime_Minutes"] <= fact_production["Planned_Time_mins"]]

# C. Fill missing Operator IDs
fact_production["Operator_ID"] = fact_production["Operator_ID"].fillna("OP_UNKNOWN")

# D. Sensor Data: Fill missing values in sensor columns using median strategy
sensor_columns = [
    "Air_Temperature_K",
    "Process_Temperature_K",
    "Humidity_Percentage",
    "Energy_Consumption_kWh"
]

for col in sensor_columns:
    fact_production[col] = (
        fact_production[col]
        .fillna(fact_production[col].median())
    )

Save Cleaned Files

In [19]:
dim_failures.to_csv(
    "Cleaned_Dim_Failure_Types.csv",
    index=False
)

dim_machines.to_csv(
    "Cleaned_Dim_Machines.csv",
    index=False
)

fact_maintenance.to_csv(
    "Cleaned_Fact_Maintenance_Events.csv",
    index=False
)

fact_production.to_csv(
    "Cleaned_Fact_Production_Log.csv",
    index=False
)

print(" All datasets cleaned and saved successfully.")

 All datasets cleaned and saved successfully.
